# Contrastive Probe Inference

Runs the spatial-grounding probe: for each cached scene whose instruction
contains a swappable spatial term, the original and term-swapped instructions
are both passed to OpenVLA against the same frame, and both predictions are
logged with `pair_id`, `role`, and `scene_id`, the schema required by the
export notebook.

Every prediction also carries the scene's `category` and `feasible_both`, so the
analysis notebook can stratify (primary target: `referent_selection` pairs judged
feasible on both sides). All categories are logged; a per-category pair-count
summary is printed before the run.

Predictions are written to a fresh CSV (`probe_predictions_v2.csv`).

**Colab GPU:** prefer L4 or A100. Requires notebook 01's install and restart to
have been done in this session, and notebook 02's cache to exist on Drive.


## 1. Mount Drive


In [23]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/openvla_cache/hf'
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/bridge_multiobj'
# v2: adds category and feasible_both columns; a fresh CSV since the header is
# fixed from the first row (see model.append_prediction_log).
PROBE_CSV = '/content/drive/MyDrive/openvla_cache/probe_predictions_v2.csv'
print('cache ->', CACHE_DIR)
print('log   ->', PROBE_CSV)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cache -> /content/drive/MyDrive/openvla_cache/bridge_multiobj
log   -> /content/drive/MyDrive/openvla_cache/probe_predictions_v2.csv


## 2. Import the code

Adds the local repository directory to the import path and clears the import
cache so a re-run picks up any edits. No GitHub clone is required.

On Colab the kernel working directory is usually `/content`, not the folder that
holds the notebook, so auto-detection also searches a mounted Drive. If that
fails, set `REPO_DIR` in the next cell to the folder that contains `model.py`.


In [24]:
import sys, os, glob, importlib

# Colab cwd is usually /content, not the notebook folder. Set REPO_DIR to the
# folder that contains model.py when auto-detection fails, for example:
#   REPO_DIR = '/content/drive/MyDrive/ECS8056'
REPO_DIR = ''


def find_repo_dir(anchor):
    """Return the directory holding `anchor`, searching several sensible roots."""
    starts = [REPO_DIR, os.getcwd()]
    nb_path = globals().get('__vsc_ipynb_file__')
    if nb_path:
        starts.insert(0, os.path.dirname(os.path.abspath(nb_path)))
    try:
        starts += [str(p) for p in (get_ipython().user_ns.get('_dh') or [])]
    except Exception:
        pass
    for known in ('/content/ECS8056', '/content/drive/MyDrive/ECS8056'):
        starts.append(known)

    seen = set()
    for start in starts:
        if not start:
            continue
        d = os.path.abspath(start)
        for _ in range(6):
            if d in seen:
                break
            seen.add(d)
            if os.path.isfile(os.path.join(d, anchor)):
                return d
            parent = os.path.dirname(d)
            if parent == d:
                break
            d = parent

    drive_root = '/content/drive/MyDrive'
    if os.path.isdir(drive_root):
        for depth in range(6):
            hits = glob.glob(os.path.join(drive_root, *(['*'] * depth), anchor))
            if hits:
                return os.path.dirname(os.path.abspath(hits[0]))
    return None


module_dir = find_repo_dir('model.py')
if module_dir is None:
    raise FileNotFoundError(
        "model.py not found. On Colab, open or upload the whole repository "
        "(not only the notebook), mount Drive, then set REPO_DIR above to the "
        f"folder that contains model.py. cwd={os.getcwd()!r}.")
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for m in ('model', 'data'):
    sys.modules.pop(m, None)
importlib.invalidate_caches()

from model import load_openvla, predict_action, run_metadata, append_prediction_log
from data import load_manifest
print('imported model.py and data.py from', module_dir)

imported model.py and data.py from /content/ECS8056


## 3. Load OpenVLA-7B


In [25]:
processor, vla, compute_dtype = load_openvla(quantize_4bit=True, precision='bf16')
meta = run_metadata(compute_dtype)
print(meta)

[load_openvla] GPU: NVIDIA A100-SXM4-80GB (sm_80, 79.3 GB) | precision=bf16 | attn=eager | 4bit=True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[load_openvla] Loaded. GPU memory allocated: 8.17 GB
{'gpu_name': 'NVIDIA A100-SXM4-80GB', 'gpu_capability': 'sm_80', 'dtype': 'bfloat16', 'seed': 42, 'torch': '2.11.0+cu128', 'transformers': '4.40.1', 'bitsandbytes': '0.50.0'}


## 4. Generate contrastive pairs from the manifest

Minimal-pair construction by antonym swap: a scene qualifies when its
instruction contains exactly one swappable spatial term, so the pair differs
in that term alone. Instructions with zero or multiple swappable terms are
excluded (a multi-term swap would change more than one relation and break the
minimal-pair property). Role `a` is always the original instruction; role `b`
is the swapped variant.


In [26]:
import re

ANTONYM_PAIRS = [
    ('in front of', 'behind'),
    ('closer to',   'farther from'),
    ('nearer to',   'farther from'),
    ('leftmost',    'rightmost'),
    ('nearest',     'farthest'),
    ('left',        'right'),
    ('top',         'bottom'),
    ('front',       'back'),   
]

SWAP = {}
for a, b in ANTONYM_PAIRS:
    SWAP.setdefault(a, b)
    SWAP.setdefault(b, a)

_KEYS = sorted(SWAP, key=len, reverse=True)
_SWAP_RE = re.compile(
    r'\b(' + '|'.join(re.escape(k) for k in _KEYS) + r')\b',
    re.IGNORECASE,
)

def make_pair(instruction: str):
    """Return (term, swapped) if exactly one swappable spatial phrase occurs."""
    hits = _SWAP_RE.findall(instruction)
    if len(hits) != 1:
        return None
    term = hits[0].lower()
    swapped = _SWAP_RE.sub(lambda m: SWAP[m.group(0).lower()], instruction)
    return term, swapped

rows = load_manifest(CACHE_DIR)
probe_set = []
for row in rows:
    made = make_pair(row['instruction'])
    if made is None:
        continue
    term, swapped = made
    probe_set.append({
        'scene_id': row['episode_index'],
        'pair_id': f"ep{int(row['episode_index']):06d}_{term}",
        'spatial_term': term,
        # Stratification fields carried through to the prediction log.
        'category': row.get('category', 'other'),
        'feasible_both': row.get('feasible_both', 'unreviewed'),
        'image_path': os.path.join(CACHE_DIR, row['image_path']),
        'instr_a': row['instruction'],
        'instr_b': swapped,
    })

print(f'{len(rows)} cached scenes -> {len(probe_set)} probe pairs')

# All categories are probed; print pair counts so the referent_selection yield
# (the primary analysis target) is visible before committing GPU time.
from collections import Counter
cat_counts = Counter(p['category'] for p in probe_set)
print('pairs per category:', dict(cat_counts))
feas = Counter(p['feasible_both'] for p in probe_set
               if p['category'] == 'referent_selection')
print('referent_selection feasible_both:', dict(feas))

for p in probe_set[:5]:
    print(f"[{p['pair_id']}] ({p['category']})")
    print('  a:', p['instr_a'])
    print('  b:', p['instr_b'])

224 cached scenes -> 72 probe pairs
pairs per category: {'placement_relation': 64, 'referent_selection': 8}
referent_selection feasible_both: {'no': 4, 'yes': 4}
[ep000000_left] (placement_relation)
  a: Place the can to the left of the pot.
  b: Place the can to the right of the pot.
[ep000002_front] (referent_selection)
  a: Slide the cloth diagonally to the front of the spoon
  b: Slide the cloth diagonally to the back of the spoon
[ep000004_right] (placement_relation)
  a: Move the kadai and place it at the right edge of the table.
  b: Move the kadai and place it at the left edge of the table.
[ep000014_top] (placement_relation)
  a: Move the Orange cloth towards the top of the table
  b: Move the Orange cloth towards the bottom of the table
[ep000017_in front of] (placement_relation)
  a: Move the colander in front of the red spoon
  b: Move the colander behind the red spoon


## 5. Run the probe

Two deterministic predictions per pair (same frame, both instructions), each
logged with the pairing columns. `sample_idx` is fixed at 0 under the
deterministic decoding strategy; the column exists so the schema does not
change if repeated sampling or paraphrase variants are added later.

Restart-safe: pairs already present in the log are skipped, so an interrupted
run resumes where it stopped.


In [27]:
import csv
import numpy as np
from PIL import Image

done = set()
if os.path.exists(PROBE_CSV):
    with open(PROBE_CSV, newline='') as f:
        done = {r['pair_id'] for r in csv.DictReader(f)}
    print(f'resuming: {len(done)} pair ids already logged')

for i, p in enumerate(probe_set):
    if p['pair_id'] in done:
        continue
    image = Image.open(p['image_path'])
    for role, instr in (('a', p['instr_a']), ('b', p['instr_b'])):
        action = predict_action(processor, vla, image, instr, compute_dtype)
        append_prediction_log(
            PROBE_CSV, action, instr, meta,
            scene_id=p['scene_id'],
            pair_id=p['pair_id'],
            role=role,
            spatial_term=p['spatial_term'],
            category=p['category'],
            feasible_both=p['feasible_both'],
            sample_idx=0,
        )
    if (i + 1) % 10 == 0:
        print(f'{i + 1}/{len(probe_set)} pairs done')

print('probe complete ->', PROBE_CSV)

resuming: 72 pair ids already logged
probe complete -> /content/drive/MyDrive/openvla_cache/probe_predictions_v2.csv


## 6. Quick directional read

A sanity read of the x-axis sign-flip rate for left/right
pairs before the export/pilot notebook runs. The formal metrics, ground-truth
validation, and frame checks belong to the next notebook.


In [28]:
import pandas as pd
log = pd.read_csv(PROBE_CSV)
lr = log[log['spatial_term'].isin(['left', 'right'])]
wide = lr.pivot_table(index='pair_id', columns='role', values='a0')
flips = (wide['a'] * wide['b'] < 0)
print(f"left/right pairs: {len(wide)} | dx sign flips: {flips.sum()} "
      f"({flips.mean():.1%})")
wide.head(8)

left/right pairs: 49 | dx sign flips: 7 (14.3%)


role,a,b
pair_id,,
ep000000_left,-0.002669,-0.002669
ep000004_right,-0.014077,-0.012958
ep000027_left,-0.002669,-0.002669
ep000031_left,-0.001327,-0.000209
ep000033_left,-0.004011,-0.004011
ep000036_right,-0.003788,-0.000209
ep000037_right,-0.001327,-0.001327
ep000043_left,-0.002446,-0.002446
